# KL-IG: Greedy Path Variants — Prototype Evaluation
ResNet50 · Greedy path ablation (SortedDim, GreedyMu, GreedyJoint) vs Linear baseline


# Setup

In [ ]:
!git clone --branch claude/general-session-FcgoB \
    https://github.com/Shameen5375/KLIG_V1.git 2>/dev/null || echo "Repo already cloned"
!pip install -q captum datasets tqdm scikit-learn


In [ ]:
import os, sys, math, json, pickle, warnings
from pathlib import Path
from collections import defaultdict
import copy

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50
from scipy import stats
from tqdm.auto import tqdm

ROOT = Path.cwd()
for candidate in [ROOT, ROOT / "infocube-main",
                  Path("/content/KLIG_V1/infocube-main"),
                  Path("/content/KLIG_V1")]:
    if (candidate / "klig").exists():
        ROOT = candidate
        break
sys.path.append(str(ROOT))

from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.core.integrator import KLIntegratedGradients
from klig.core.path import LinearPath
from klig.core.greedy_path import SortedDimPath, GreedyMuAttributor, GreedyJointAttributor

warnings.filterwarnings("ignore", category=UserWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"Root:   {ROOT}")


In [ ]:
# ── Sample sizes ──────────────────────────────────────────────────────
N_IMGS    = 1000
N_subset  = 100
N_STEPS   = 50
N_SAMPLES = 10
TIER_A    = 1000

# ── Attribution hyperparameters ────────────────────────────────────────
BLUR_SIGMA  = 16.0
BLUR_KERNEL = 51
IG_STEPS    = 50
EG_SAMPLES  = 50
SG_SAMPLES  = 50
BIG_STEPS   = 50
BIG_SIGMA   = 10.0

# ── Greedy-path hyperparameters ────────────────────────────────────────
GAMMA_LO           = 0.25   # SortedDim: γ for highest-gradient dim
GAMMA_HI           = 4.0    # SortedDim: γ for lowest-gradient dim
SORTED_DIM_SAMPLES = 32     # MC samples for prior gradient estimation
SIGMA_FINAL        = 1 / 256
ADAPTIVE_SIGMA     = True

# ── Metric hyperparameters ─────────────────────────────────────────────
N_INSERTION_STEPS   = 50
N_SENS_SUBSETS      = 30
SENS_FRACTIONS      = [0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.8]
PERTURBATION_SIGMAS = [0.01, 0.02, 0.05, 0.1, 0.2]
PERTURBATION_RUNS   = 3
OCCLUSION_PATCH     = 14
OCCLUSION_STRIDE    = 7
OCCLUSION_RATIOS    = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

# ── ImageNet normalisation ─────────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

HF_DATASET_NAME = "evanarlian/imagenet_1k_resized_256"
HF_SPLIT = "val"

# ── Persistent cache ───────────────────────────────────────────────────
USE_DRIVE       = True
DRIVE_CACHE     = "/content/drive/MyDrive/klig_greedy_cache"
LOCAL_CACHE     = "greedy_eval_cache"
FORCE_RECOMPUTE = False
if USE_DRIVE:
    try:
        from google.colab import drive
        if not Path("/content/drive").exists() or not any(Path("/content/drive").iterdir()):
            drive.mount("/content/drive")
        CACHE_DIR = Path(DRIVE_CACHE)
        print(f"[cache] Drive: {CACHE_DIR}")
    except Exception as e:
        CACHE_DIR = Path(LOCAL_CACHE)
        print(f"[cache] Drive unavailable ({e}) — falling back to {CACHE_DIR}")
else:
    CACHE_DIR = Path(LOCAL_CACHE)
    print(f"[cache] Local: {CACHE_DIR}")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ── Methods + colors ───────────────────────────────────────────────────
methods_all = [
    "KL-IG (adaptive)",
    "KL-IG-SortedDim",
    "KL-IG-GreedyMu",
    "KL-IG-GreedyJoint",
]

COLORS_ALL = {
    "KL-IG (adaptive)":  "#1B5E3F",
    "KL-IG-SortedDim":   "#2196F3",
    "KL-IG-GreedyMu":    "#4CAF50",
    "KL-IG-GreedyJoint": "#FF5722",
}

print(f"N_IMGS={N_IMGS}, methods={len(methods_all)}")
print(f"Cache: {CACHE_DIR.resolve()}")

In [ ]:
def load_model():
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(DEVICE).eval()
    return model, weights.meta["categories"]

def denormalize(x):
    mean = torch.tensor(IMAGENET_MEAN, device=x.device).view(-1, 1, 1)
    std  = torch.tensor(IMAGENET_STD,  device=x.device).view(-1, 1, 1)
    if x.dim() == 4:
        mean, std = mean.unsqueeze(0), std.unsqueeze(0)
    return (x * std + mean).clamp(0, 1)

def get_sigma_final(model, x, target):
    if ADAPTIVE_SIGMA:
        return min(max(find_sigma_stop(model, x, target=target, tau=0.95), 1.0/256.0), 1.0)
    return SIGMA_FINAL

model, imagenet_labels = load_model()
print(f"Model: ResNet50, {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")


In [ ]:
def load_imagenet_subset(n_images):
    try:
        from datasets import load_dataset
        print(f"[dataset] HuggingFace {HF_DATASET_NAME} [{HF_SPLIT}]")
        ds = load_dataset(HF_DATASET_NAME, split=HF_SPLIT, streaming=True) \
             .shuffle(seed=42, buffer_size=50_000)
        seen, out = set(), []
        for ex in ds:
            y = int(ex["label"])
            if y in seen:
                continue
            seen.add(y)
            out.append((ex["image"].convert("RGB"), y))
            if len(out) >= n_images:
                break
        return out
    except Exception as e:
        print(f"[dataset] Failed: {e}")
        return []

raw_samples = load_imagenet_subset(TIER_A)
dataset = []
with torch.no_grad():
    for i, (pil, gt) in enumerate(tqdm(raw_samples, desc="prep")):
        x = TRANSFORM(pil).unsqueeze(0).to(DEVICE)
        probs = model(x).softmax(-1)[0]
        top1  = int(probs.argmax())
        target = gt if probs[gt].item() > 0.05 else top1
        dataset.append({"idx": i, "x": x, "target": target,
                         "label_str": imagenet_labels[target]})

eval_idx = list(range(min(N_IMGS, len(dataset))))
print(f"Dataset: {len(dataset)} images, evaluating {len(eval_idx)}")


In [ ]:
def absmax_collapse(a):
    if a.dim() == 4: a = a.squeeze(0)
    idx = a.abs().argmax(dim=0, keepdim=True)
    return a.gather(0, idx).squeeze(0)

# attrs stored in all_attrs are already (H, W) — to_heatmap only clips/normalises
def to_heatmap(attr_hw, clip_pct=99.0):
    m = attr_hw.cpu().float()
    clip = float(torch.quantile(m.abs(), clip_pct / 100.0))
    m = m.clamp(-clip, clip)
    lo, hi = m.min(), m.max()
    if hi > lo:
        m = (m - lo) / (hi - lo)
    return m.numpy()


# Attribution loop

In [ ]:
_cache = CACHE_DIR / "greedy_attrs.pkl"

if not FORCE_RECOMPUTE and _cache.exists():
    with open(_cache, "rb") as f:
        all_attrs = pickle.load(f)
    print(f"[cache] Loaded {len(all_attrs)} results from {_cache}")
else:
    all_attrs = defaultdict(dict)   # all_attrs[method][img_idx] = attr (H,W) tensor

    for row in tqdm(dataset, desc="images"):
        idx    = row["idx"]
        x      = row["x"]          # (1, 3, 224, 224)
        target = row["target"]
        sigma  = get_sigma_final(model, x, target)
        x1     = x.squeeze(0)      # (3, 224, 224)

        # ── KL-IG (adaptive, linear path) ──
        ig = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, path=LinearPath(), device=DEVICE
        )
        res = ig.attribute(x1, target=target)
        all_attrs["KL-IG (adaptive)"][idx] = absmax_collapse(res.attr).cpu()

        # ── KL-IG-SortedDim ──
        sp = SortedDimPath.from_model_and_input(
            model, x1, target=target,
            n_samples=SORTED_DIM_SAMPLES,
            gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI,
        )
        ig_s = KLIntegratedGradients(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, path=sp, device=DEVICE
        )
        res_s = ig_s.attribute(x1, target=target)
        all_attrs["KL-IG-SortedDim"][idx] = absmax_collapse(res_s.attr).cpu()

        # ── KL-IG-GreedyMu ──
        gmu = GreedyMuAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, device=DEVICE
        )
        res_gmu = gmu.attribute(x1, target=target)
        all_attrs["KL-IG-GreedyMu"][idx] = absmax_collapse(res_gmu.attr).cpu()

        # ── KL-IG-GreedyJoint ──
        gjoint = GreedyJointAttributor(
            model, n_steps=N_STEPS, n_samples=N_SAMPLES,
            sigma_final=sigma, device=DEVICE
        )
        res_gj = gjoint.attribute(x1, target=target)
        all_attrs["KL-IG-GreedyJoint"][idx] = absmax_collapse(res_gj.attr).cpu()

    with open(_cache, "wb") as f:
        pickle.dump(dict(all_attrs), f)
    print(f"[cache] Saved to {_cache}")

# 1. Single-image attribution maps

In [ ]:
# Pick a representative image (first in dataset)
row = dataset[0]
x_vis  = row["x"]          # (1, 3, 224, 224)
target_vis = row["target"]
x_display  = denormalize(x_vis[0]).cpu().permute(1, 2, 0).numpy()

fig, axes = plt.subplots(1, len(methods_all) + 1,
                          figsize=(3 * (len(methods_all) + 1), 3.5))
axes[0].imshow(x_display)
axes[0].set_title(f'Input\n{row["label_str"]}', fontsize=9)
axes[0].axis("off")

for ax, m in zip(axes[1:], methods_all):
    attr = all_attrs[m][row["idx"]]
    hm = to_heatmap(attr)
    ax.imshow(hm, cmap="RdBu_r", vmin=0, vmax=1)
    ax.set_title(m, fontsize=8)
    ax.axis("off")

plt.suptitle("Attribution maps — absmax collapse, 99th-pct clip", fontsize=11)
plt.tight_layout()
plt.show()


# 2. Sparsity (Gini coefficient)

In [ ]:
_cache_g = CACHE_DIR / "greedy_gini.pkl"

def gini(v):
    v = v.abs().flatten().sort()[0].float()
    n = len(v)
    idx = torch.arange(1, n + 1, dtype=torch.float)
    return float((2 * (idx * v).sum() / (n * v.sum() + 1e-12) - (n + 1) / n).item())

if not FORCE_RECOMPUTE and _cache_g.exists():
    with open(_cache_g, "rb") as f:
        all_gini = pickle.load(f)
    print(f"[cache] Loaded from {_cache_g}")
else:
    all_gini = {m: [gini(all_attrs[m][i]) for i in eval_idx] for m in methods_all}
    with open(_cache_g, "wb") as f:
        pickle.dump(all_gini, f)

ci95 = lambda v: 1.96 * np.std(v) / (len(v) ** 0.5)
means = [np.mean(all_gini[m]) for m in methods_all]
cis   = [ci95(all_gini[m])   for m in methods_all]

fig, ax = plt.subplots(figsize=(10, 4), facecolor="white")
for xi, (m, mu, ci) in enumerate(zip(methods_all, means, cis)):
    ax.bar(xi, mu, color=COLORS_ALL[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
ax.set_xticks(range(len(methods_all)))
ax.set_xticklabels(methods_all, rotation=25, ha="right", fontsize=9)
ax.set_ylabel("Gini coefficient (higher = sparser)")
ax.set_title(f"Attribution sparsity — Gini (n={len(eval_idx)} images)")
plt.tight_layout()
plt.show()


# 3. Insertion / Deletion AUC

In [ ]:
_cache_id = CACHE_DIR / "greedy_ins_del.pkl"

def insertion_deletion(model, x, attr_map, target,
                        n_steps=N_INSERTION_STEPS, batch_size=64):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    n_pix = H * W
    order = attr_map.detach().view(-1).argsort(descending=True)
    pps = max(1, n_pix // n_steps)
    blur_base = F.avg_pool2d(x, kernel_size=31, stride=1, padding=15)
    ins_scores, del_scores = [], []
    x_ins = blur_base.clone()
    x_del = x.clone()
    with torch.no_grad():
        for step in range(n_steps):
            pixels = order[step * pps: (step + 1) * pps]
            for ch in range(C):
                flat_ins = x_ins[:, ch].reshape(-1)
                flat_del = x_del[:, ch].reshape(-1)
                flat_ins[pixels] = x[:, ch].reshape(-1)[pixels]
                flat_del[pixels] = blur_base[:, ch].reshape(-1)[pixels]
            ins_scores.append(model(x_ins).softmax(-1)[0, target].item())
            del_scores.append(model(x_del).softmax(-1)[0, target].item())
    ins_auc = float(np.trapz(ins_scores) / n_steps)
    del_auc = float(np.trapz(del_scores) / n_steps)
    return ins_auc, del_auc

if not FORCE_RECOMPUTE and _cache_id.exists():
    with open(_cache_id, "rb") as f:
        ins_auc, del_auc = pickle.load(f)
    print(f"[cache] Loaded from {_cache_id}")
else:
    ins_auc = defaultdict(list)
    del_auc = defaultdict(list)
    for row in tqdm(dataset, desc="ins/del"):
        idx = row["idx"]
        x   = row["x"]
        t   = row["target"]
        for m in methods_all:
            attr = all_attrs[m][idx].to(DEVICE).unsqueeze(0)
            i_auc, d_auc = insertion_deletion(model, x, attr, t)
            ins_auc[m].append(i_auc)
            del_auc[m].append(d_auc)
    with open(_cache_id, "wb") as f:
        pickle.dump((dict(ins_auc), dict(del_auc)), f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor="white")
for ax, (title, aucs) in zip(axes, [("Insertion AUC ↑", ins_auc),
                                      ("Deletion AUC ↓", del_auc)]):
    means_id = [np.mean(aucs[m]) for m in methods_all]
    cis_id   = [1.96 * np.std(aucs[m]) / len(aucs[m]) ** 0.5 for m in methods_all]
    for xi, (m, mu, ci) in enumerate(zip(methods_all, means_id, cis_id)):
        ax.bar(xi, mu, color=COLORS_ALL[m], alpha=0.85, width=0.6)
        ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
    ax.set_xticks(range(len(methods_all)))
    ax.set_xticklabels(methods_all, rotation=25, ha="right", fontsize=9)
    ax.set_title(title)
plt.suptitle(f"Insertion / Deletion AUC (n={len(eval_idx)} images)", fontsize=12)
plt.tight_layout()
plt.show()


# 4. Sensitivity-n

In [ ]:
_cache_sn = CACHE_DIR / "greedy_sens_n.pkl"

def sensitivity_n(model, x, attr_map, target, n_subsets=N_SENS_SUBSETS,
                   fractions=SENS_FRACTIONS):
    C, H, W = x.shape[1], x.shape[2], x.shape[3]
    n_pix  = H * W
    f0     = model(x).softmax(-1)[0, target].item()
    flat_a = attr_map.detach().view(-1).abs().cpu().numpy()
    pccs   = []
    with torch.no_grad():
        for frac in fractions:
            k = max(1, int(n_pix * frac))
            attr_sums, score_drops = [], []
            for _ in range(n_subsets):
                subset = np.random.choice(n_pix, k, replace=False)
                attr_sums.append(flat_a[subset].sum())
                x_mask = x.clone()
                for ch in range(C):
                    x_mask[:, ch].reshape(-1)[subset] = 0.0
                score_drops.append(f0 - model(x_mask).softmax(-1)[0, target].item())
            r, _ = stats.pearsonr(attr_sums, score_drops)
            pccs.append(float(r) if not np.isnan(r) else 0.0)
    return pccs

if not FORCE_RECOMPUTE and _cache_sn.exists():
    with open(_cache_sn, "rb") as f:
        all_sens = pickle.load(f)
    print(f"[cache] Loaded from {_cache_sn}")
else:
    all_sens = defaultdict(lambda: [[] for _ in SENS_FRACTIONS])
    for row in tqdm(dataset, desc="sens-n"):
        idx = row["idx"]
        x   = row["x"]
        t   = row["target"]
        for m in methods_all:
            attr = all_attrs[m][idx].to(DEVICE).unsqueeze(0)
            pccs = sensitivity_n(model, x, attr, t)
            for fi, p in enumerate(pccs):
                all_sens[m][fi].append(p)
    with open(_cache_sn, "wb") as f:
        pickle.dump(dict(all_sens), f)

fig, ax = plt.subplots(figsize=(10, 5), facecolor="white")
fracs = np.array(SENS_FRACTIONS)
for m in methods_all:
    means_sn = [np.mean(all_sens[m][fi]) for fi in range(len(fracs))]
    ax.plot(fracs, means_sn, "-o", color=COLORS_ALL[m], label=m, lw=2)
ax.set_xlabel("Subset fraction n")
ax.set_ylabel("Pearson correlation")
ax.set_title(f"Sensitivity-n PCC (n={len(eval_idx)} images)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 5. Object Focus Ratio (OFR)

In [ ]:
from PIL import Image as PILImage
from datasets import load_dataset as _hf_load

N_OFR_IMGS   = 200
_cache_ofr   = CACHE_DIR / "greedy_ofr.pkl"
_cache_ofr_l = Path("greedy_ofr.pkl")

_MASK_TRANSFORM = T.Compose([
    T.Resize(256, interpolation=T.InterpolationMode.NEAREST),
    T.CenterCrop(224),
])

def _load_imagenet_s_mask(mask_pil):
    m  = np.array(mask_pil)
    fg = PILImage.fromarray((m[:, :, 0] > 0).astype(np.uint8) * 255, mode="L")
    return (np.array(_MASK_TRANSFORM(fg)) > 0).astype(np.uint8)

def object_focus_ratio(attr_hw, obj_mask):
    a = attr_hw.detach().cpu().numpy() if hasattr(attr_hw, "detach") else np.asarray(attr_hw)
    a = np.abs(a)
    total = a.sum()
    return float(a[obj_mask == 1].sum() / total) if total > 1e-12 else 0.0

def _try_load_ofr():
    for p in (_cache_ofr, _cache_ofr_l):
        if p.exists():
            try:
                with open(p, "rb") as f:
                    return pickle.load(f)
            except Exception:
                pass
    return None

all_ofr = None if FORCE_RECOMPUTE else _try_load_ofr()

if all_ofr is None:
    print(f"Computing OFR over up to {N_OFR_IMGS} ImageNet-S images ...")
    all_ofr = {m: [] for m in methods_all}
    count   = 0

    ds_seg = _hf_load("braceletboy/imagenet-s", split="validation", streaming=True)
    for ex in tqdm(ds_seg, desc="OFR", total=N_OFR_IMGS):
        if count >= N_OFR_IMGS:
            break
        tgt = int(ex["label"])
        if not (0 <= tgt < 1000):
            continue
        obj_mask = _load_imagenet_s_mask(ex["mask"])
        frac = obj_mask.mean()
        if frac < 0.02 or frac > 0.85:
            continue
        x = TRANSFORM(ex["image"].convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            if model(x).argmax(1).item() != tgt:
                continue
        sf = get_sigma_final(model, x, tgt)
        x1 = x.squeeze(0)

        method_fns = {
            "KL-IG (adaptive)": lambda: object_focus_ratio(
                absmax_collapse(KLIntegratedGradients(
                    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                    sigma_final=sf, path=LinearPath(), device=DEVICE,
                ).attribute(x1, target=tgt).attr), obj_mask),

            "KL-IG-SortedDim": lambda: object_focus_ratio(
                absmax_collapse(KLIntegratedGradients(
                    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                    sigma_final=sf, device=DEVICE,
                    path=SortedDimPath.from_model_and_input(
                        model, x1, target=tgt,
                        n_samples=SORTED_DIM_SAMPLES,
                        gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI),
                ).attribute(x1, target=tgt).attr), obj_mask),

            "KL-IG-GreedyMu": lambda: object_focus_ratio(
                absmax_collapse(GreedyMuAttributor(
                    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                    sigma_final=sf, device=DEVICE,
                ).attribute(x1, target=tgt).attr), obj_mask),

            "KL-IG-GreedyJoint": lambda: object_focus_ratio(
                absmax_collapse(GreedyJointAttributor(
                    model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                    sigma_final=sf, device=DEVICE,
                ).attribute(x1, target=tgt).attr), obj_mask),
        }

        for m, fn in method_fns.items():
            try:
                all_ofr[m].append(fn())
            except Exception as e:
                print(f"  [{m}] img {count}: {type(e).__name__}: {e}")

        count += 1

    print("List lengths:", {m: len(all_ofr[m]) for m in methods_all})
    for p in (_cache_ofr_l, _cache_ofr):
        try:
            with open(p, "wb") as f:
                pickle.dump(all_ofr, f)
            print(f"[cache] Saved to {p}")
        except Exception as e:
            print(f"[cache] Save failed {p}: {e}")

# ── Plot ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4), facecolor="white")
for xi, m in enumerate(methods_all):
    v = all_ofr[m]
    if not v:
        ax.text(xi, 0.02, "no data", ha="center", fontsize=8, color="red")
        continue
    mu = np.mean(v)
    ci = 1.96 * np.std(v) / math.sqrt(len(v))
    ax.bar(xi, mu, color=COLORS_ALL[m], alpha=0.85, width=0.6)
    ax.errorbar(xi, mu, yerr=ci, fmt="none", color="black", capsize=4, lw=1.5)
    ax.text(xi, mu + 0.01, f"n={len(v)}", ha="center", fontsize=7)
ax.set_xticks(range(len(methods_all)))
ax.set_xticklabels(methods_all, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Object Focus Ratio ↑")
ax.set_title("OFR — fraction of |attr| inside GT object mask")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

# 6. Greedy path trajectories (PCA, single image)

In [ ]:
# Re-run a single image with diagnostic output to visualise trajectories
from sklearn.decomposition import PCA

row_vis = dataset[0]
x1_vis  = row_vis["x"].squeeze(0)
tgt_vis = row_vis["target"]
sigma_vis = get_sigma_final(model, row_vis["x"], tgt_vis)

sp_vis = SortedDimPath.from_model_and_input(
    model, x1_vis, target=tgt_vis,
    n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)

gmu_vis = GreedyMuAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                              sigma_final=sigma_vis, device=DEVICE)
res_gmu_vis = gmu_vis.attribute(x1_vis, target=tgt_vis)

gjoint_vis = GreedyJointAttributor(model, n_steps=N_STEPS, n_samples=N_SAMPLES,
                                    sigma_final=sigma_vis, device=DEVICE)
res_gj_vis = gjoint_vis.attribute(x1_vis, target=tgt_vis)

mu_final_np = x1_vis.detach().cpu().reshape(-1).numpy()
ts_np = np.linspace(0.5/N_STEPS, 1.0 - 0.5/N_STEPS, N_STEPS)
gamma_np = sp_vis._gamma.cpu().reshape(-1).numpy()

linear_wps  = np.outer(ts_np, mu_final_np)
sorted_wps  = np.array([ts_np[k]**gamma_np * mu_final_np for k in range(N_STEPS)])
gmu_wps     = torch.stack(res_gmu_vis.waypoints_mu).reshape(N_STEPS, -1).cpu().numpy()
gjoint_wps  = torch.stack(res_gj_vis.waypoints_mu).reshape(N_STEPS, -1).cpu().numpy()

all_wps = np.concatenate([linear_wps, sorted_wps, gmu_wps, gjoint_wps])
pca = PCA(n_components=2).fit(all_wps)

fig, ax = plt.subplots(figsize=(7, 6), facecolor="white")
for label, wps in [
    ("KL-IG (adaptive)",  linear_wps),
    ("KL-IG-SortedDim",   sorted_wps),
    ("KL-IG-GreedyMu",    gmu_wps),
    ("KL-IG-GreedyJoint", gjoint_wps),
]:
    pts = pca.transform(wps)
    ax.plot(pts[:, 0], pts[:, 1], "-o", color=COLORS_ALL[label],
            markersize=3, lw=1.5, label=label, alpha=0.85)

origin = pca.transform(np.zeros((1, len(mu_final_np))))
target_pt = pca.transform(mu_final_np.reshape(1, -1))
ax.scatter(*origin.T, marker="*", s=200, color="black", zorder=5, label="Prior")
ax.scatter(*target_pt.T, marker="D", s=100, color="gold", edgecolors="black",
            zorder=5, label="μ_final")

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("μ-space trajectories: PCA projection")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 7. Cumulative path attribution (front-load vs back-load)

In [ ]:
T_STEPS_PLOT = [0.25, 0.5, 0.75, 1.0]
VIS_IMG_IDX  = 0   # change to visualise a different image

row_vis  = dataset[VIS_IMG_IDX]
x_vis    = row_vis["x"]
tgt_vis  = row_vis["target"]
sig_vis  = get_sigma_final(model, x_vis, tgt_vis)
img_orig = np.clip(denormalize(x_vis[0]).detach().cpu().permute(1,2,0).numpy(), 0, 1)


def _cumulative_path_maps(method, x, target, sigma, t_fracs, n_steps, n_samples, device):
    """
    Cumulative (H,W) attribution at each path fraction.
    Returns {frac: (H,W) tensor} for each frac in t_fracs.
    """
    x1   = x.squeeze(0).to(device)
    x_sh = x1.shape
    D    = x1.numel()
    mu_f = x1.detach()
    lv_f = torch.full_like(mu_f, 2.0 * math.log(sigma))
    obj  = lambda xs: model(xs)[:, target].mean()

    snap_at = {f: max(1, round(f * n_steps)) for f in t_fracs}
    snaps   = {}
    attr_mu = torch.zeros_like(mu_f)
    attr_lv = torch.zeros_like(mu_f)

    saved = [p.requires_grad for p in model.parameters()]
    for p in model.parameters():
        p.requires_grad_(False)

    try:
        if method in ("KL-IG (adaptive)", "KL-IG-SortedDim"):
            path = (LinearPath() if method == "KL-IG (adaptive)" else
                    SortedDimPath.from_model_and_input(
                        model, x1, target=target,
                        n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI))
            ts = path.steps(n_steps).tolist()
            dt = 1.0 / n_steps
            for k, t in enumerate(ts):
                mu_t, lv_t = path.at(t, mu_f, lv_f)
                dm, dl     = path.derivatives(t, mu_f, lv_f)
                mu_g = mu_t.detach().requires_grad_(True)
                lv_g = lv_t.detach().requires_grad_(True)
                eps  = torch.randn(n_samples, *x_sh, device=device)
                xs   = mu_g.unsqueeze(0) + (0.5 * lv_g).exp().unsqueeze(0) * eps
                obj(xs).backward()
                attr_mu += mu_g.grad * dm * dt
                attr_lv += lv_g.grad * dl * dt
                for f, s in snap_at.items():
                    if k + 1 == s:
                        snaps[f] = absmax_collapse((attr_mu + attr_lv).clone()).cpu()

        elif method == "KL-IG-GreedyMu":
            mu_c = torch.zeros_like(mu_f)
            for k in range(n_steps):
                lv_c = ((k + 0.5) / n_steps) * lv_f
                mu_g = mu_c.detach().requires_grad_(True)
                lv_g = lv_c.detach().requires_grad_(True)
                eps  = torch.randn(n_samples, *x_sh, device=device)
                xs   = mu_g.unsqueeze(0) + (0.5 * lv_g).exp().unsqueeze(0) * eps
                obj(xs).backward()
                g_mu, g_lv = mu_g.grad.clone(), lv_g.grad.clone()
                rem  = n_steps - k
                d_mu = (D * g_mu.abs() / (g_mu.abs().sum() + 1e-12) * (mu_f - mu_c) / rem
                        if k < n_steps - 1 else mu_f - mu_c)
                d_lv = lv_f / n_steps
                attr_mu += g_mu * d_mu
                attr_lv += g_lv * d_lv
                mu_c = (mu_c + d_mu).detach()
                for f, s in snap_at.items():
                    if k + 1 == s:
                        snaps[f] = absmax_collapse((attr_mu + attr_lv).clone()).cpu()

        elif method == "KL-IG-GreedyJoint":
            mu_c = torch.zeros_like(mu_f)
            lv_c = torch.zeros_like(lv_f)
            for k in range(n_steps):
                mu_g = mu_c.detach().requires_grad_(True)
                lv_g = lv_c.detach().requires_grad_(True)
                eps  = torch.randn(n_samples, *x_sh, device=device)
                xs   = mu_g.unsqueeze(0) + (0.5 * lv_g).exp().unsqueeze(0) * eps
                obj(xs).backward()
                g_mu, g_lv = mu_g.grad.clone(), lv_g.grad.clone()
                rem = n_steps - k
                if k < n_steps - 1:
                    jt   = g_mu.abs() + g_lv.abs()
                    w    = D * jt / (jt.sum() + 1e-12)
                    d_mu = w * (mu_f - mu_c) / rem
                    d_lv = w * (lv_f - lv_c) / rem
                else:
                    d_mu = mu_f - mu_c
                    d_lv = lv_f - lv_c
                attr_mu += g_mu * d_mu
                attr_lv += g_lv * d_lv
                mu_c = (mu_c + d_mu).detach()
                lv_c = (lv_c + d_lv).detach()
                for f, s in snap_at.items():
                    if k + 1 == s:
                        snaps[f] = absmax_collapse((attr_mu + attr_lv).clone()).cpu()

    finally:
        for p, s in zip(model.parameters(), saved):
            p.requires_grad_(s)

    return snaps


print("Computing cumulative attribution maps ...")
maps = {}
for m in tqdm(methods_all, desc="methods"):
    maps[m] = _cumulative_path_maps(
        m, x_vis, tgt_vis, sig_vis, T_STEPS_PLOT, N_STEPS, N_SAMPLES, DEVICE)

# ── Plot ────────────────────────────────────────────────────────────────
N_COLS = 1 + len(T_STEPS_PLOT)
N_ROWS = len(methods_all)

fig, axes = plt.subplots(N_ROWS, N_COLS,
                          figsize=(2.4 * N_COLS, 2.4 * N_ROWS),
                          facecolor="white")
if N_ROWS == 1:
    axes = axes[np.newaxis, :]

for mi, method in enumerate(methods_all):
    full_a = maps[method][1.0].numpy()
    vmax   = max(float(np.percentile(np.abs(full_a), 99)), 1e-12)

    ax0 = axes[mi, 0]
    ax0.imshow(img_orig)
    ax0.set_xticks([]); ax0.set_yticks([])
    for sp in ax0.spines.values():
        sp.set_visible(False)
    if mi == 0:
        ax0.set_title("Original", fontsize=10, fontweight="bold")
    ax0.set_ylabel(method, fontsize=9, rotation=0,
                    labelpad=90, va="center", fontweight="bold")

    for ci, t in enumerate(T_STEPS_PLOT):
        ax = axes[mi, 1 + ci]
        ax.imshow(maps[method][t].numpy(), cmap="cividis", vmin=-vmax, vmax=vmax)
        ax.axis("off")
        if mi == 0:
            ax.set_title(f"attr [0→{t:.2f}]", fontsize=10, fontweight="bold")

plt.suptitle(
    f"Cumulative path attribution — {imagenet_labels[tgt_vis]}\n"
    "Front-loaded: salient regions appear early   |   Back-loaded: appear late",
    fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

# 8. Path-attribution change curves

In [ ]:
N_CURVE_IMGS = 20   # images to average over

_cache_pc = CACHE_DIR / "greedy_path_curves.pkl"


def _path_step_signal(path, model, x1, target_idx, sigma, n_steps, n_samples, device):
    """Walk a DistributionPath and record |g_μ·dμ + g_lv·dlv| at each step."""
    model.eval()
    saved = [p.requires_grad for p in model.parameters()]
    for p in model.parameters():
        p.requires_grad_(False)

    x_shape = x1.shape
    logvar_final = torch.full_like(x1, 2.0 * math.log(sigma))
    mu_final = x1.detach().to(device)
    objective = lambda xs: model(xs)[:, target_idx].mean()
    ts = path.steps(n_steps).tolist()
    dt = 1.0 / n_steps
    signals = []

    try:
        for t in ts:
            mu_t, lv_t = path.at(t, mu_final, logvar_final)
            dmu_dt, dlv_dt = path.derivatives(t, mu_final, logvar_final)

            mu_t_g = mu_t.detach().requires_grad_(True)
            lv_t_g = lv_t.detach().requires_grad_(True)

            eps = torch.randn(n_samples, *x_shape, device=device)
            std_t = (0.5 * lv_t_g).exp()
            x_samp = mu_t_g.unsqueeze(0) + std_t.unsqueeze(0) * eps
            out = objective(x_samp)
            out.backward()

            g_mu = mu_t_g.grad.clone()
            g_lv = lv_t_g.grad.clone()
            sig = float((g_mu * dmu_dt * dt + g_lv * dlv_dt * dt).abs().sum().item())
            signals.append(sig)
    finally:
        for p, s in zip(model.parameters(), saved):
            p.requires_grad_(s)

    return np.array(signals)


def get_change(method, x, target_idx, sigma, n_steps, n_samples, device):
    x1 = x.squeeze(0).to(device)
    alphas = np.linspace(0, 1, n_steps)
    if method == "KL-IG (adaptive)":
        sig = _path_step_signal(
            LinearPath(), model, x1, target_idx, sigma, n_steps, n_samples, device)
        return alphas, sig
    if method == "KL-IG-SortedDim":
        sp = SortedDimPath.from_model_and_input(
            model, x1, target=target_idx,
            n_samples=SORTED_DIM_SAMPLES, gamma_lo=GAMMA_LO, gamma_hi=GAMMA_HI)
        sig = _path_step_signal(
            sp, model, x1, target_idx, sigma, n_steps, n_samples, device)
        return alphas, sig
    if method == "KL-IG-GreedyMu":
        r = GreedyMuAttributor(
            model, n_steps=n_steps, n_samples=n_samples,
            sigma_final=sigma, device=device).attribute(x1, target=target_idx)
        return alphas, np.array(r.step_grad_signal)
    if method == "KL-IG-GreedyJoint":
        r = GreedyJointAttributor(
            model, n_steps=n_steps, n_samples=n_samples,
            sigma_final=sigma, device=device).attribute(x1, target=target_idx)
        return alphas, np.array(r.step_grad_signal)
    raise ValueError(f"Unknown method: {method}")


PATH_METHODS = ["KL-IG (adaptive)", "KL-IG-SortedDim", "KL-IG-GreedyMu", "KL-IG-GreedyJoint"]

if not FORCE_RECOMPUTE and _cache_pc.exists():
    with open(_cache_pc, "rb") as f:
        curve_signals = pickle.load(f)
    print(f"[cache] Loaded from {_cache_pc}")
else:
    curve_signals = {m: [] for m in PATH_METHODS}

    img_pool = []
    for row in dataset[:N_CURVE_IMGS]:
        x, tgt = row["x"], row["target"]
        sf = get_sigma_final(model, x, tgt)
        img_pool.append((row, sf))

    for d, sf in tqdm(img_pool, desc="path-curves"):
        x, tgt = d["x"], d["target"]
        for m in PATH_METHODS:
            _, sig = get_change(m, x, tgt, sf, N_STEPS, N_SAMPLES, DEVICE)
            curve_signals[m].append(sig)

    with open(_cache_pc, "wb") as f:
        pickle.dump(curve_signals, f)
    print(f"[cache] Saved to {_cache_pc}")

# ── Plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor="white")
xs = np.linspace(0, 1, N_STEPS)

for m in PATH_METHODS:
    mat = np.stack(curve_signals[m])          # (N_CURVE_IMGS, N_STEPS)
    mean = mat.mean(axis=0)
    se   = mat.std(axis=0) / math.sqrt(len(mat))

    # Left: absolute signal, log scale
    axes[0].semilogy(xs, mean, color=COLORS_ALL[m], lw=2, label=m)
    axes[0].fill_between(xs, np.maximum(mean - se, 1e-12), mean + se,
                          color=COLORS_ALL[m], alpha=0.15)

    # Right: normalised cumulative fraction (front-loading = faster rise)
    frac = mean / (mean.sum() + 1e-12)
    axes[1].plot(xs, np.cumsum(frac), color=COLORS_ALL[m], lw=2, label=m)

axes[0].set_xlabel("Integration step α")
axes[0].set_ylabel("|Δattribution| (log scale)")
axes[0].set_title(f"Attribution change per step\n(mean ± SE, n={N_CURVE_IMGS} images)")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0.5, color="gray", lw=0.8, ls="--", label="50% mark")
axes[1].set_xlabel("Integration step α")
axes[1].set_ylabel("Cumulative fraction of total |Δattr|")
axes[1].set_title("Signal front-loading\n(steeper early rise = more front-loaded)")
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Path-attribution change curves — KL-IG variants", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# 10. 2D Synthetic Function Probe

Visualise how each KL-IG variant distributes attribution on analytic 2-D functions.
**Rows** = test function; **Col 0** = f(x₁,x₂) truth; **Cols 1-4** = attribution magnitude
(√(a₁²+a₂²)) with sparse quiver arrows showing direction.
No model or data needed — fully self-contained, runs ~2 min on GPU.

In [ ]:
import math, torch, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from pathlib import Path

# ── config ───────────────────────────────────────────────────────────────────
PROBE_GRID  = 32        # grid resolution per axis  (32×32 = 1024 pts)
PROBE_STEPS = 20        # integration steps
PROBE_SAMPS = 64        # MC samples per gradient estimate
PROBE_BATCH = 64        # grid points evaluated in one forward pass
GAMMA_LO    = 0.3       # SortedDim: exponent for highest-gradient dim
GAMMA_HI    = 3.5       # SortedDim: exponent for lowest-gradient dim
_DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"probe device: {_DEV}")

# ── analytic test functions  (input: (N,2) → (N,)) ──────────────────────────
PROBE_FUNCS = {
    "Checkerboard": lambda x: torch.tanh(
        4 * torch.sin(math.pi * x[:, 0]) * torch.sin(math.pi * x[:, 1])),
    "Diagonal":     lambda x: torch.sin(math.pi * (x[:, 0] + x[:, 1])),
    "Radial":       lambda x: torch.sin(
        2.5 * (x[:, 0].pow(2) + x[:, 1].pow(2)).clamp(min=1e-8).sqrt()),
    "Point":        lambda x: torch.exp(-25 * (x[:, 0].pow(2) + x[:, 1].pow(2))),
}

# ── MC gradient helper (batched: mu/lv shape (N,D)) ──────────────────────────
def _mc_grads_batch(fn, mu, lv, n_samps):
    """Returns g_mu, g_lv both shape (N,D)."""
    N, D = mu.shape
    mu2 = mu.detach().requires_grad_(True)
    lv2 = lv.detach().requires_grad_(True)
    eps = torch.randn(n_samps, N, D, device=mu.device)
    # (S,N,D) -> (S*N, D)
    x = (mu2.unsqueeze(0) + (0.5 * lv2).exp().unsqueeze(0) * eps).reshape(n_samps * N, D)
    f = fn(x).reshape(n_samps, N).mean(0)  # (N,)
    gm, gl = torch.autograd.grad(f.sum(), [mu2, lv2])
    return gm.detach(), gl.detach()   # (N,D)

# ── attribution kernels (batched, returns (N,D) tensors) ─────────────────────
def _attr_linear_batch(fn, pts):
    N, D = pts.shape
    mu  = pts.to(_DEV)
    lv  = torch.zeros_like(mu)
    acc = torch.zeros_like(mu)
    dt  = 1.0 / PROBE_STEPS
    for s in range(1, PROBE_STEPS + 1):
        t = (s - 0.5) * dt
        gm, _ = _mc_grads_batch(fn, t * mu, t * lv, PROBE_SAMPS)
        acc   = acc + gm * mu * dt
    return acc.cpu()

def _attr_sorted_batch(fn, pts):
    N, D = pts.shape
    mu  = pts.to(_DEV)
    lv  = torch.zeros_like(mu)
    # one-shot prior gradient (use first point as proxy; recompute per-point for accuracy)
    prior_mu = torch.zeros_like(mu)
    gm_prior, _ = _mc_grads_batch(fn, prior_mu, lv, PROBE_SAMPS)
    mag    = gm_prior.abs()                             # (N,D)
    # rank dims per sample
    order  = mag.argsort(dim=1, descending=True).argsort(dim=1).float()
    gamma  = GAMMA_LO + (GAMMA_HI - GAMMA_LO) * order / max(D - 1, 1)  # (N,D)
    acc = torch.zeros_like(mu)
    dt  = 1.0 / PROBE_STEPS
    for s in range(1, PROBE_STEPS + 1):
        t     = s * dt
        t_prv = (s - 1) * dt
        mu_t  = (t   ** gamma) * mu
        mu_p  = (t_prv ** gamma) * mu
        lv_t  = t * lv
        dm    = mu_t - mu_p
        gm, _ = _mc_grads_batch(fn, mu_t, lv_t, PROBE_SAMPS)
        acc   = acc + gm * dm
    return acc.cpu()

def _attr_greedy_mu_batch(fn, pts):
    N, D    = pts.shape
    mu_fin  = pts.to(_DEV)
    lv_fin  = torch.zeros_like(mu_fin)
    mu_c    = torch.zeros_like(mu_fin)
    lv_c    = torch.zeros_like(mu_fin)
    acc     = torch.zeros_like(mu_fin)
    for step in range(1, PROBE_STEPS + 1):
        rem    = PROBE_STEPS - step + 1
        gm, _  = _mc_grads_batch(fn, mu_c, lv_c, PROBE_SAMPS)
        w      = D * gm.abs() / (gm.abs().sum(dim=1, keepdim=True) + 1e-8)
        dm     = w * (mu_fin - mu_c) / rem
        dl     = (lv_fin - lv_c) / rem
        acc    = acc + gm * dm
        mu_c   = mu_c + dm
        lv_c   = lv_c + dl
    return acc.cpu()

def _attr_greedy_joint_batch(fn, pts):
    N, D    = pts.shape
    mu_fin  = pts.to(_DEV)
    lv_fin  = torch.zeros_like(mu_fin)
    mu_c    = torch.zeros_like(mu_fin)
    lv_c    = torch.zeros_like(mu_fin)
    acc     = torch.zeros_like(mu_fin)
    for step in range(1, PROBE_STEPS + 1):
        rem    = PROBE_STEPS - step + 1
        gm, gl = _mc_grads_batch(fn, mu_c, lv_c, PROBE_SAMPS)
        score  = gm.abs() + gl.abs()
        w      = D * score / (score.sum(dim=1, keepdim=True) + 1e-8)
        dm     = w * (mu_fin - mu_c) / rem
        dl     = w * (lv_fin - lv_c) / rem
        acc    = acc + gm * dm
        mu_c   = mu_c + dm
        lv_c   = lv_c + dl
    return acc.cpu()

ATTR_BATCH_FNS = {
    "KL-IG\n(linear)":    _attr_linear_batch,
    "KL-IG\nSortedDim":   _attr_sorted_batch,
    "KL-IG\nGreedyMu":    _attr_greedy_mu_batch,
    "KL-IG\nGreedyJoint": _attr_greedy_joint_batch,
}

# ── build grid ───────────────────────────────────────────────────────────────
_xs  = torch.linspace(-1.0, 1.0, PROBE_GRID)
g1, g2 = torch.meshgrid(_xs, _xs, indexing="xy")
grid_pts = torch.stack([g1.flatten(), g2.flatten()], dim=1)  # (G², 2)
G2 = PROBE_GRID * PROBE_GRID

# ── compute (store full vectors for quiver) ───────────────────────────────────
print(f"Grid {PROBE_GRID}×{PROBE_GRID}={G2} pts, {PROBE_STEPS} steps, {PROBE_SAMPS} samps")
results = {}   # results[fname][aname] = {"vec": (G,G,2), "mag": (G,G)}
for fname, fn in PROBE_FUNCS.items():
    results[fname] = {}
    for aname, afn in ATTR_BATCH_FNS.items():
        vecs = []
        for b0 in range(0, G2, PROBE_BATCH):
            batch = grid_pts[b0:b0 + PROBE_BATCH]
            vecs.append(afn(fn, batch))
        v = torch.cat(vecs, dim=0).reshape(PROBE_GRID, PROBE_GRID, 2).numpy()
        mag = np.sqrt(v[..., 0]**2 + v[..., 1]**2)
        results[fname][aname] = {"vec": v, "mag": mag}
    print(f"  {fname} done")

# ── quiver subsample (every K grid points) ───────────────────────────────────
QK = 4   # show arrow every QK cells
qi  = np.arange(QK // 2, PROBE_GRID, QK)
qx  = _xs.numpy()[qi]

# ── plot ─────────────────────────────────────────────────────────────────────
n_rows = len(PROBE_FUNCS)
n_cols = 1 + len(ATTR_BATCH_FNS)
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(2.6 * n_cols, 2.6 * n_rows),
                         constrained_layout=True)

col_headers = ["f(x₁,x₂)"] + [k.replace("\n", " ") for k in ATTR_BATCH_FNS]
for c, h in enumerate(col_headers):
    axes[0, c].set_title(h, fontsize=8.5, fontweight="bold", pad=4)

ext = [-1, 1, -1, 1]
for r, (fname, fn) in enumerate(PROBE_FUNCS.items()):
    axes[r, 0].set_ylabel(fname, fontsize=8.5, fontweight="bold")

    # col 0: true function value
    with torch.no_grad():
        fv = fn(grid_pts).numpy().reshape(PROBE_GRID, PROBE_GRID)
    axes[r, 0].imshow(fv, origin="lower", cmap="RdBu_r", extent=ext,
                      vmin=-1, vmax=1, interpolation="bicubic", aspect="equal")
    axes[r, 0].set_xticks([-1, 0, 1]); axes[r, 0].set_yticks([-1, 0, 1])
    axes[r, 0].tick_params(labelsize=6)

    # cols 1-4: attribution magnitude + quiver
    for c, aname in enumerate(ATTR_BATCH_FNS.keys(), start=1):
        mag = results[fname][aname]["mag"]
        vec = results[fname][aname]["vec"]
        vmax = np.percentile(mag, 97) + 1e-9
        im = axes[r, c].imshow(mag, origin="lower", cmap="inferno", extent=ext,
                               norm=Normalize(vmin=0, vmax=vmax),
                               interpolation="bicubic", aspect="equal")
        # quiver overlay (subsample)
        qv = vec[np.ix_(qi, qi)]               # (Nq,Nq,2)
        qmag = np.sqrt(qv[..., 0]**2 + qv[..., 1]**2) + 1e-9
        axes[r, c].quiver(qx, qx,
                          qv[..., 0] / qmag, qv[..., 1] / qmag,
                          alpha=0.55, color="white", scale=14,
                          width=0.006, headwidth=3, headlength=4)
        axes[r, c].set_xticks([-1, 0, 1]); axes[r, c].set_yticks([-1, 0, 1])
        axes[r, c].tick_params(labelsize=6)

fig.suptitle("Attribution magnitude  ‖a(x)‖  per KL-IG variant", fontsize=10, y=1.01)

_probe_path = Path("klig_probe_2d.png")
fig.savefig(_probe_path, dpi=160, bbox_inches="tight")
print(f"Saved → {_probe_path.resolve()}")
plt.show()

# 9. Summary table

In [ ]:
import pandas as pd

ci95 = lambda v: 1.96 * np.std(v) / (len(v) ** 0.5)

rows = []
for m in methods_all:
    row = {"Method": m}
    row["Gini ↑"]       = f"{np.mean(all_gini[m]):.4f} ± {ci95(all_gini[m]):.4f}"
    row["Ins AUC ↑"]    = f"{np.mean(ins_auc[m]):.4f} ± {ci95(ins_auc[m]):.4f}"
    row["Del AUC ↓"]    = f"{np.mean(del_auc[m]):.4f} ± {ci95(del_auc[m]):.4f}"
    sn_mean = np.mean([np.mean(all_sens[m][fi]) for fi in range(len(SENS_FRACTIONS))])
    row["Sens-n PCC ↑"] = f"{sn_mean:.4f}"
    if "all_ofr" in globals() and m in all_ofr and all_ofr[m]:
        row["OFR ↑"]    = f"{np.mean(all_ofr[m]):.4f} ± {ci95(all_ofr[m]):.4f}"
    else:
        row["OFR ↑"]    = "—"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Method")
print(df.to_string())